""" script for data generation for the purpuse of teaching network finding lindbladian coefficients """

# Importing libraries 

importing necessary libraries and declaring necessary constants 

In [1]:
import numpy as np
import torch
import torch.nn as nn
import numpy as np
import json

from pathlib import Path

import matplotlib.pyplot as plt

hbar = 1.054571817 * 1e-34
eV = 1.6 * 1e-19
ps = 1e-12

# Lindbladian solver class 

Class containnig whole procidure for creating Lindbladian (that defines density matrix evolution), initializing and performing density matrix evolution

## rho fixing function
function compacating numerical artefacts that destort hermitivity of density matrix 

In [4]:
#funkcja naprawiająca numeryczne błędy w maicerzy gestosci po integracji rungego-kutty
def FixRho(rho):
    # Handle both 2D (N,N) and 3D batched (B,N,N) inputs
    squeeze = rho.ndim == 2
    if squeeze:
        rho = rho.unsqueeze(0)

    # 1. Force Hermitian (more aggressively)
    rho_dag = rho.conj().transpose(1, 2)
    rho = (rho + rho_dag) / 2
    # Add tiny regularization to diagonal to help convergence
    eps = 1e-10
    eye = torch.eye(rho.shape[-1], dtype=rho.dtype, device=rho.device)
    rho = rho + eps * eye.unsqueeze(0)

    # 2. Batched eigen-decomposition
    eigvals, eigvecs = torch.linalg.eigh(rho)

    # 3. Clip negative eigenvalues
    eigvals = torch.clamp(eigvals, min=0)

    # 4. Reconstruct:  V @ diag(λ) @ V†
    rho_fixed = eigvecs @ torch.diag_embed(eigvals.to(eigvecs.dtype)) @ eigvecs.conj().transpose(1, 2)
    # 5. Normalize trace

    trace = rho_fixed.diagonal(dim1=1, dim2=2).sum(dim=1)
    rho_fixed = rho_fixed / trace[:, None, None]

    return rho_fixed.squeeze(0) if squeeze else rho_fixed

#losowanie liczby zespolonej o module z przedziału [0, A] i fazie z przedziału [0, 2*pi]
def rand_cplx(A):
    r = np.random.uniform(0, A)          # moduł
    phi = np.random.uniform(0, 2*np.pi)  # faza

    return r * np.exp(1j * phi)

## lindbladian class

In [6]:
"""LINDBLAD SOLVER FOR ARBITRARY NUMBER OF QUBITS"""
# klasa rozwiązująca równanie Lindblada dla gęstości ρ:
class Lindblad_solver:
    """
    Solves the Lindblad master equation for the density matrix ρ:

        dρ/dt = -i[H, ρ] + Σ_k γ_k D[L_k](ρ)

    where D[L](ρ) = L ρ L† - ½ {L†L, ρ}  (dissipator)
    """

    def __init__(self, H, L_ops, gammas, dt):
        """
        H      : Hamiltonian (NxN complex array)
        L_ops  : list of jump operators [L_k], each (NxN)
        gammas : list of decay rates [g_k], one per jump operator
        dt     : time step
        """
        self.H      = torch.tensor(np.array(H, dtype=np.complex64))  # complex64, not complex128
        self.L_ops  = [torch.tensor(np.array(L, dtype=np.complex64)) for L in L_ops]
        self.gammas = list(gammas)
        self.dt     = dt
        self.LdagL  = [L.conj().mT @ L for L in self.L_ops]

#lindblad_rhs computes the right-hand side of the Lindblad equation for a given ρ
    def lindblad_rhs(self, rho):
        H = self.H.to(device=rho.device, dtype=rho.dtype)
        drho = -1j/hbar * (H @ rho - rho @ H)

        for gamma, Lk, LdLk in zip(self.gammas, self.L_ops, self.LdagL):
            Lk = Lk.to(device=rho.device, dtype=rho.dtype)
            LdLk = LdLk.to(device=rho.device, dtype=rho.dtype)
            drho += gamma * (Lk @ rho @ Lk.conj().mT - 0.5 * (LdLk @ rho + rho @ LdLk))
        return drho
# renormalize ensures that the trace of ρ remains 1 after each integration step to suppress floating-point drift.
    def _renormalize(self, rho):
        """Enforce Tr(ρ) = 1 after each step to suppress floating-point drift."""
        return rho / np.trace(rho)

    # RK4 step
# step advances ρ by one time step using classical RK4 and returns the next ρ.
    def step(self, rho):
        """
        Advance ρ by one time step using classical RK4.
        Returns rho_{n+1}.
        """
        dt = self.dt
        L  = self.lindblad_rhs

        k1 = L(rho)
        k2 = L(rho + 0.5 * dt * k1)
        k3 = L(rho + 0.5 * dt * k2)
        k4 = L(rho +       dt * k3)

        rho_next = rho + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        return FixRho(rho_next)

    # Full trajectory
# run_simulation propagates from an initial ρ0 for n_steps time steps and returns the trajectory as an array of shape (n_steps+1, N, N).
    def run_simulation(self, rho0, n_steps=1000):
        """
        Propagate from rho0 for n_steps time steps.
        Returns array of shape (n_steps+1, N, N).
        """
        rho = torch.tensor(np.array(rho0, dtype=complex))
        rhos = [rho.clone()]

        times = [0]

        for i in range(n_steps):
            rho = self.step(rho)
            rhos.append(rho.clone())
            times.append((i + 1) * dt)
            #print(np.amin(np.linalg.eigvals(rho.cpu().numpy())))

        return [np.array(times), np.array(rhos)]

    def populations(rhos):
        """Extract diagonal populations from a trajectory array."""
        return np.real(np.einsum('tii->ti', rhos))   # shape (T, N)
    

def convert_rhos_to_numpy(rhos):

    rhos_np = np.array(rhos)

    if rhos_np.ndim == 4:
        rhos_np = rhos_np.squeeze(1)

    return rhos_np

## ploting sygnals

In [8]:
# plotting signals (as many as wanted on one plot)
def plot_signals(signals, t_axis = None, labels=None, points=None, pointlabels=None):
    fig, ax = plt.subplots()
    for s, sig in enumerate(signals):
        if t_axis is None:
            t_axis = np.linspace(0.,1., num=len(sig), endpoint=True)
        ax.plot(t_axis, sig, label=labels[s])
    if points is not None:
        for p, pts in enumerate(points):
            ax.scatter(pts[0], pts[1], marker='x', label=pointlabels[p])
    if labels is not None or pointlabels is not None: ax.legend()
    plt.show()

# File management functions 

functions that create files, manage process of saving data and reading data from the files 

## generating parameters functions

In [11]:
def generate_initial_state(coeff_1=1/np.sqrt(2), coeff_2=0, coeff_3=0, coeff_4=1/np.sqrt(2), max_amplitude=0.1):
    """
    Generate a Bell-like initial state.

    max_amplitude:
        maximal admixture of |01> and |10>
    """

    eps1 = np.random.uniform(-max_amplitude, max_amplitude)
    eps2 = np.random.uniform(-max_amplitude, max_amplitude)
    eps3 = np.random.uniform(-max_amplitude, max_amplitude)
    eps4 = np.random.uniform(-max_amplitude, max_amplitude)

    psi = np.array([coeff_1+eps1, coeff_2+eps2, coeff_3+eps3, coeff_4+eps4], dtype=complex)
    psi = psi / np.linalg.norm(psi)

    return psi

In [12]:
def generate_parameters(
        tau_relax1_range,
        tau_relax2_range,
        tau_deph1_range,
        tau_deph2_range,
        tau_cordeph_range):

    taus = {
        "tau_relax1": tau0_relax1 * np.random.uniform(*tau_relax1_range),
        "tau_relax2": tau0_relax2 * np.random.uniform(*tau_relax2_range),
        "tau_deph1": tau0_deph1 * np.random.uniform(*tau_deph1_range),
        "tau_deph2": tau0_deph2 * np.random.uniform(*tau_deph2_range),
        "tau_cordeph": tau0_cordeph * np.random.uniform(*tau_cordeph_range)}

    return taus

## saving/loading a particular trajectory function

In [15]:
def save_trajectory(
        filename,
        psi0,
        taus,
        times,
        rhos):

    np.savez_compressed(
        filename,

        psi0=psi0,

        tau_relax1=taus["tau_relax1"],
        tau_relax2=taus["tau_relax2"],
        tau_deph1=taus["tau_deph1"],
        tau_deph2=taus["tau_deph2"],
        tau_cordeph=taus["tau_cordeph"],

        times=times,
        rhos=rhos
    )

In [16]:
def load_trajectory(filename):

    data = np.load(filename)

    taus = {
        "tau_relax1": float(data["tau_relax1"]),
        "tau_relax2": float(data["tau_relax2"]),
        "tau_deph1": float(data["tau_deph1"]),
        "tau_deph2": float(data["tau_deph2"]),
        "tau_cordeph": float(data["tau_cordeph"])
    }

    return {"psi0": data["psi0"], "taus": taus, "times": data["times"], "rhos": data["rhos"]}

## creating dataset description file 

creating .json file with parameters describing properties of particular dataset like parameters' ranges etc. 

In [25]:
def create_metadata(
        dataset_dir,
        metadata):

    metadata_file = Path(dataset_dir) / "metadata.json"

    with open(metadata_file, "w") as f:

        json.dump(
            metadata,
            f,
            indent=4
        )

## generating whole dataset 

In [28]:
def generate_dataset(
        dataset_name,
        n_trajectories,

        max_amplitude,

        tau_relax1_range,
        tau_relax2_range,

        tau_deph1_range,
        tau_deph2_range,

        tau_cordeph_range):

    dataset_dir = Path(dataset_name)
    dataset_dir.mkdir( parents=True, exist_ok=True)
    
    metadata = {
        "n_trajectories": n_trajectories,

        "initial_state": {
            "coeff_1": coeff_1,
            "coeff_2": coeff_2,
            "coeff_3": coeff_3,
            "coeff_4": coeff_4,
            "max_amplitude": max_amplitude
        },
        "tau0_relax1": {
            "tau0": tau0_relax1,
            "range": tau_relax1_range
        },

        "tau0_relax2": {
            "tau0": tau0_relax2,
            "range": tau_relax2_range
        },

        "tau0_deph1": {
            "tau0": tau0_deph1,
            "range": tau_deph1_range
        },

        "tau0_deph2": {
            "tau0": tau0_deph2,
            "range": tau_deph2_range
        },

        "tau0_cordeph": {
            "tau0": tau0_cordeph,
            "range": tau_cordeph_range
        }
    }

    create_metadata(dataset_dir, metadata)

    for i in range(n_trajectories):

        psi0_np = generate_initial_state(coeff_1=coeff_1, coeff_2=coeff_2, coeff_3=coeff_3, coeff_4=coeff_4, max_amplitude=max_amplitude)
        psi0 = torch.tensor(psi0_np, dtype=torch.complex64)
        rho0 = torch.outer(psi0, psi0.conj()).unsqueeze(0)

        taus = generate_parameters(tau_relax1_range, tau_relax2_range, tau_deph1_range, tau_deph2_range, tau_cordeph_range)
        gamma_relax1 = 1 / taus["tau_relax1"]
        gamma_relax2 = 1 / taus["tau_relax2"]
        gamma_deph1  = 1 / taus["tau_deph1"]
        gamma_deph2  = 1 / taus["tau_deph2"]
        gamma_cordeph = 1 / taus["tau_cordeph"]
        gammas = [gamma_relax1, gamma_relax2, gamma_deph1, gamma_deph2, gamma_cordeph]
        #################################################################
        # TU WSTAWIASZ SWOJĄ EWOLUCJĘ
        #################################################################

        solver = Lindblad_solver(H, L_ops=L, gammas=gammas, dt=dt)  
        [times, rhos]  = solver.run_simulation(rho0, n_steps=n_steps)
        rhos_np = convert_rhos_to_numpy(rhos)
        #################################################################

        filename = dataset_dir / f"traj_{i:06d}.npz"

        save_trajectory(
            filename,
            psi0_np,
            taus,
            times,
            rhos_np
        )

        if i % 100 == 0:

            print(
                f"{i}/{n_trajectories} trajectories generated"
            )

# Implementation

## system parameters

In [32]:
# 2 qubit - dissipation

J = 0 * 1e-3 * eV       # (meV)
dt = 0.1 * ps      # (ps)
n_steps = 400

H = np.array([
    [1,  0,  0, 0],
    [0, -1,  2, 0],
    [0,  2, -1, 0],
    [0,  0,  0, 1]
], dtype=complex)*J/4


I2 = np.eye(2, dtype=complex)
# lindbladian relaksacji 
L1 = np.array([[0, 1],
               [0, 0]], dtype=complex) #relaks

#lindbladian dephasowania
L2 = np.array([[1,  0],
               [0, -1]], dtype=complex) #dephas

#tworzenie lindbladianu - funkcja np.kron - iloczyn kroneckera - tworzy operatory dla 2-qubitowego systemu, gdzie L1 działa na pierwszy qubit, a I2 na drugi, i odwrotnie, oraz dla dephasowania i korelacji dephasowania
L = [np.kron(L1, I2), np.kron(I2, L1), np.kron(L2, I2), np.kron(I2, L2), np.kron(I2, L2)+np.kron(L2, I2)]



## generation

In [34]:
#initial state - Bell state
coeff_1 = 1/np.sqrt(2)
coeff_2 = 0
coeff_3 = 0
coeff_4 = 1/np.sqrt(2)

#linbladian parameters
tau0_relax1 = 60 * ps
tau0_relax2 = 60 * ps
tau0_deph1 = 60 * ps 
tau0_deph2 = 60 * ps
tau0_cordeph = 60 * ps

generate_dataset( #function that generates the dataset by simulating trajectories of the Lindblad equation with random initial states and parameters, and saves them to disk in a structured format.

    dataset_name="dataset_6_60ps_narrow_5000t", # name 

    n_trajectories=5000, # how many trajectories to generate

    max_amplitude=0.1, # maximal distortion of the initial state from the Bell state (it is randomly generated in the range [0, max_amplitude])

    tau_relax1_range=(0.8, 1.2), # maximal variation of the relaxation time tau_relax1 from its base value tau0_relax1 (it is randomly generated)
    tau_relax2_range=(0.8, 1.2), # maximal variation of the relaxation time tau_relax2 

    tau_deph1_range=(0.8, 1.2), # -||- for the dephasing time tau_deph1
    tau_deph2_range=(0.8, 1.2), # -||- for the dephasing time tau_deph2

    tau_cordeph_range=(0.8, 1.2) # -||- for the correlation dephasing time tau_cordeph
)

generate_dataset( #function that generates the dataset by simulating trajectories of the Lindblad equation with random initial states and parameters, and saves them to disk in a structured format.

    dataset_name="dataset_7_60ps_mid_5000t", # name 

    n_trajectories=5000, # how many trajectories to generate

    max_amplitude=0.1, # maximal distortion of the initial state from the Bell state (it is randomly generated in the range [0, max_amplitude])

    tau_relax1_range=(0.5, 2.0), # maximal variation of the relaxation time tau_relax1 from its base value tau0_relax1 (it is randomly generated)
    tau_relax2_range=(0.5, 2.0), # maximal variation of the relaxation time tau_relax2 

    tau_deph1_range=(0.5, 2.0), # -||- for the dephasing time tau_deph1
    tau_deph2_range=(0.5, 2.0), # -||- for the dephasing time tau_deph2

    tau_cordeph_range=(0.5, 2.0) # -||- for the correlation dephasing time tau_cordeph
)

generate_dataset( #function that generates the dataset by simulating trajectories of the Lindblad equation with random initial states and parameters, and saves them to disk in a structured format.

    dataset_name="dataset_8_60ps_wide_5000t", # name 

    n_trajectories=5000, # how many trajectories to generate

    max_amplitude=0.1, # maximal distortion of the initial state from the Bell state (it is randomly generated in the range [0, max_amplitude])

    tau_relax1_range=(0.25, 4.0), # maximal variation of the relaxation time tau_relax1 from its base value tau0_relax1 (it is randomly generated)
    tau_relax2_range=(0.25, 4.0), # maximal variation of the relaxation time tau_relax2 

    tau_deph1_range=(0.25, 4.0), # -||- for the dephasing time tau_deph1
    tau_deph2_range=(0.25, 4.0), # -||- for the dephasing time tau_deph2

    tau_cordeph_range=(0.25, 4.0) # -||- for the correlation dephasing time tau_cordeph
)

C:\Users\Ziemowit\AppData\Local\Temp\ipykernel_30960\345279885.py:65: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  rho = torch.tensor(np.array(rho0, dtype=complex))


0/5000 trajectories generated
100/5000 trajectories generated
200/5000 trajectories generated
300/5000 trajectories generated
400/5000 trajectories generated
500/5000 trajectories generated
600/5000 trajectories generated
700/5000 trajectories generated
800/5000 trajectories generated
900/5000 trajectories generated
1000/5000 trajectories generated
1100/5000 trajectories generated
1200/5000 trajectories generated
1300/5000 trajectories generated
1400/5000 trajectories generated
1500/5000 trajectories generated
1600/5000 trajectories generated
1700/5000 trajectories generated
1800/5000 trajectories generated
1900/5000 trajectories generated
2000/5000 trajectories generated
2100/5000 trajectories generated
2200/5000 trajectories generated
2300/5000 trajectories generated
2400/5000 trajectories generated
2500/5000 trajectories generated
2600/5000 trajectories generated
2700/5000 trajectories generated
2800/5000 trajectories generated
2900/5000 trajectories generated
3000/5000 trajectories